In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import zipfile
import os
import sys
from lib.const import *

In [2]:
# Download the 4 csv physical data

df_1 = pd.read_csv(f'{CLEANED_DATA_DIR}/Physical dataset/phy_att_1_cleaned.csv', delimiter=',', header=0, encoding='utf-8')
df_2 = pd.read_csv(f'{CLEANED_DATA_DIR}/Physical dataset/phy_att_2_cleaned.csv', delimiter=',', header=0, encoding='utf-8')
df_3 = pd.read_csv(f'{CLEANED_DATA_DIR}/Physical dataset/phy_att_3_cleaned.csv', delimiter=',', header=0, encoding='utf-8')
df_4 = pd.read_csv(f'{CLEANED_DATA_DIR}/Physical dataset/phy_att_4_cleaned.csv', delimiter=',', header=0, encoding='utf-8')

In [3]:
# Remove Pump_3 column and Valv_1 to Valv_9 columns, because they are all set to False

columns_to_remove = ['Time','Pump_3', 'Valv_1', 'Valv_2', 'Valv_3', 'Valv_4', 'Valv_5', 'Valv_6', 'Valv_7', 'Valv_8', 'Valv_9']

df_1 = df_1.drop(columns=columns_to_remove)
df_2 = df_2.drop(columns=columns_to_remove)
df_3 = df_3.drop(columns=columns_to_remove)
df_4 = df_4.drop(columns=columns_to_remove)

In [4]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, f1_score, balanced_accuracy_score, matthews_corrcoef
)
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
import pandas as pd
import plotly.express as px
import numpy as np
import time

In [5]:
# Evaluation function 
def evaluation_model(y_test, y_pred):
    # Évaluation
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.2f}")
    # print(f"F1-Score: {f1_score(y_test, y_pred):.2f}")
    print(f"MCC: {matthews_corrcoef(y_test, y_pred):.2f}")

In [ ]:
def knn_model(df, index, k_max=20):
    # Préparer les données
    features = df.drop(['Label', 'Label_n'], axis=1)
    target = df['Label']

    # Normalisation des features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Gestion du déséquilibre des classes avec SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(features_scaled, target)

    # Division des données
    X_train, X_test, y_train, y_test = train_test_split(
        X_resampled, y_resampled, test_size=0.3, random_state=42
    )

    results = []  # Pour stocker les résultats
    confusion_matrices = {}  # Pour stocker les matrices de confusion
    total_training_time = 0  # Pour calculer le temps moyen d'entraînement

    # Boucle sur les valeurs de k
    for k in range(1, k_max + 1):
        knn = rs(n_neighbors=k)

        # Mesure du temps d'entraînement
        start_time = time.time()
        knn.fit(X_train, y_train)
        training_time = time.time() - start_time
        total_training_time += training_time

        # Prédictions
        y_pred = knn.predict(X_test)

        # Calcul des métriques
        accuracy = accuracy_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)

        # Sauvegarde des résultats
        results.append({"k": k, "accuracy": accuracy, "training_time": training_time})
        confusion_matrices[k] = cm

    # Temps d'entraînement moyen
    train_time_avg = total_training_time / k_max

    # Convertir les résultats en DataFrame
    results_df = pd.DataFrame(results)

    # Créer un graphique de la précision en fonction de k
    fig = px.line(
        results_df,
        x="k",
        y="accuracy",
        title=f"Précision en fonction de k (KNN) pour DataFrame {index}",
        labels={"k": "Nombre de voisins (k)", "accuracy": "Précision"},
        markers=True,
    )

    # Sauvegarder le graphique
    output_dir = FIGURE_DIR
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    fig_path = os.path.join(output_dir, f"knn_accuracy_df_{index}.png")
    fig.write_image(fig_path)


    print(f"Graphique sauvegardé à : {fig_path}")
    print(f"Temps d'entraînement moyen : {train_time_avg:.4f} secondes")

    return confusion_matrices


In [7]:
# KNN on all df
confusion_matrices_1 = knn_model(df_1,1)
confusion_matrices_2 = knn_model(df_2,2)
confusion_matrices_3 = knn_model(df_3,3)
confusion_matrices_4 = knn_model(df_4,4)


RuntimeError: Kaleido now requires that chrome/chromium is installed separately. Kaleido will try to detect it automatically, but the environmental error "BROWSER_PATH" can also be set

In [ ]:
# Affichage d'une matrice de confusion spécifique (exemple pour k=5)
selected_k = 5
if selected_k in confusion_matrices:
    cm = confusion_matrices[selected_k]
    fig_cm = px.imshow(
        cm,
        text_auto=".2f",
        color_continuous_scale="rdylbu",
        labels=dict(x="Predicted", y="True", color="Count"),
        title=f"Matrice de confusion pour k={selected_k}",
    )
    fig_cm.show()

# Save the img for k=5
fig_cm.write_image("data/results/figures/knn_cm_df_4.png")

NameError: name 'confusion_matrices' is not defined

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, f1_score, balanced_accuracy_score, matthews_corrcoef
)

df = df_1

# Diviser les données en caractéristiques (X) et cible (y)
X = df.drop(columns=['Label_n', 'Label'])  # Features
y = df['Label']  # Target (0 ou 1)

# Diviser en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Créer et entraîner le modèle CART
cart_model = DecisionTreeClassifier(random_state=42, max_depth=5)
cart_model.fit(X_train, y_train)

# Prédictions
y_pred = cart_model.predict(X_test)

# Évaluation
print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.2f}")
# print(f"F1-Score: {f1_score(y_test, y_pred):.2f}")
print(f"MCC: {matthews_corrcoef(y_test, y_pred):.2f}")

# Matrice de confusion
conf_matrix = confusion_matrix(y_test, y_pred)
fig = px.imshow(
    conf_matrix,
    text_auto="int",
    labels=dict(x="Predicted", y="True", color="Count"),
    title="Matrice de confusion (CART)",
    color_continuous_scale="rdylbu"
)
fig.show()

KeyboardInterrupt: 